# 10. 침묵 vs 초기 반응 그룹 비교 분석

**분석 목적:** 유저의 첫 반응을 이끌어내기 위한 최소한의 기획적 조건을 데이터로 찾는다.

**사용 데이터:**
- `data/preprocessed/steam_indie_games.csv` — 초기 반응 그룹 (리뷰 10개 이상, 9,169개)
- `data/preprocessed/steam_indie_games_silence.csv` — 침묵 그룹 (리뷰 0~9개, 동일 장르 필터 적용)

**분석 흐름:**
1. 두 그룹 규모 및 분포 요약
2. 주요 장르 분포 비교
3. 태그 특이도 (Lift) 비교
4. 출시 가격 비교
5. 종합 해석 — 유저 첫 반응을 이끌어내기 위한 최소 기획 조건

In [1]:
import ast
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print('라이브러리 로드 완료')

라이브러리 로드 완료


## 1. 데이터 로드 및 그룹 분류

In [2]:
DATA_DIR = Path('../../../data/preprocessed')

df_response = pd.read_csv(DATA_DIR / 'steam_indie_games.csv')
df_silence  = pd.read_csv(DATA_DIR / 'steam_indie_games_silence.csv')

df_response['response_group'] = '초기 반응 (≥10개)'
df_silence['response_group']  = '침묵 (<10개)'

df = pd.concat([df_response, df_silence], ignore_index=True)

print(f'초기 반응 그룹 : {len(df_response):,}개')
print(f'침묵 그룹      : {len(df_silence):,}개')
print(f'전체           : {len(df):,}개')

초기 반응 그룹 : 8,730개
침묵 그룹      : 6,676개
전체           : 15,406개


In [3]:
TARGET_GENRES = ['Action', 'Adventure', 'Casual', 'RPG', 'Simulation', 'Strategy', 'Sports', 'Racing']

GROUP_COLORS = {
    '초기 반응 (≥10개)': '#4C72B0',
    '침묵 (<10개)':      '#C44E52',
}
GROUP_ORDER = ['침묵 (<10개)', '초기 반응 (≥10개)']


def parse_genres(value):
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(g).strip() for g in parsed if str(g).strip()]
    except (ValueError, SyntaxError):
        pass
    return [g.strip() for g in str(value).split(',') if g.strip()]


def parse_tags(value):
    if pd.isna(value):
        return {}
    try:
        return json.loads(value)
    except (json.JSONDecodeError, TypeError):
        return {}


def top_tags(tag_dict, n=5):
    return sorted(tag_dict, key=tag_dict.get, reverse=True)[:n]


df['genre_list'] = df['genres'].apply(parse_genres)
df['tag_dict']   = df['tags'].apply(parse_tags)
df['top_tags']   = df['tag_dict'].apply(lambda d: top_tags(d, n=5))

print('파싱 완료')

파싱 완료


## 2. 그룹별 규모 요약

In [4]:
group_summary = (
    df.groupby('response_group')
    .agg(
        게임수=('appid', 'count'),
        리뷰수_중앙값=('total_reviews', 'median'),
        리뷰수_평균=('total_reviews', 'mean'),
        가격_중앙값=('price', lambda x: (x[x > 0]).median()),
    )
    .round(1)
)
group_summary['비율(%)'] = (group_summary['게임수'] / len(df) * 100).round(1)
group_summary = group_summary.reindex(GROUP_ORDER)

display(group_summary)

,게임수,리뷰수_중앙값,리뷰수_평균,가격_중앙값,비율(%)
response_group,,,,,
침묵 (<10개),6676,3.0,3.8,4.0,43.3
초기 반응 (≥10개),8730,40.0,774.1,6.7,56.7


In [5]:
fig = px.bar(
    group_summary.reset_index(),
    x='response_group',
    y='게임수',
    color='response_group',
    color_discrete_map=GROUP_COLORS,
    text='게임수',
    title='그룹별 게임 수 (2023~2025년, EA·F2P·부적합 장르 제외)',
    labels={'response_group': '그룹', '게임수': '게임 수'},
    category_orders={'response_group': GROUP_ORDER},
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(showlegend=False, height=420)
fig.show()

**해석:** 두 그룹의 비율은 출시 후 최소한의 유저 반응(리뷰 10개)조차 얻지 못한 게임이 얼마나 되는지를 보여준다. 이후 분석에서 두 그룹 간 장르·태그·가격 차이를 통해 첫 반응을 이끌어내는 요인을 찾는다.

## 분석 1. 주요 장르 분포 비교

각 그룹에서 주요 장르가 차지하는 비율을 비교한다. 초기 반응 그룹에서 과대표현된 장르가 유저 반응을 이끌어내기 유리한 장르다.

In [6]:
df_genre = (
    df.explode('genre_list')
    .rename(columns={'genre_list': 'genre'})
    .query('genre in @TARGET_GENRES')
    .copy()
)

# 그룹별 전체 게임 수 (장르 중복 카운트 기준)
group_total = df_genre.groupby('response_group')['appid'].nunique()

genre_dist = (
    df_genre.groupby(['response_group', 'genre'])['appid']
    .nunique()
    .reset_index(name='game_count')
)
genre_dist['비율(%)'] = genre_dist.apply(
    lambda r: r['game_count'] / group_total[r['response_group']] * 100, axis=1
).round(1)

# 초기 반응 그룹 기준 장르 정렬
genre_order = (
    genre_dist[genre_dist['response_group'] == '초기 반응 (≥10개)']
    .sort_values('비율(%)', ascending=False)['genre']
    .tolist()
)

fig = px.bar(
    genre_dist,
    x='genre',
    y='비율(%)',
    color='response_group',
    color_discrete_map=GROUP_COLORS,
    barmode='group',
    text='비율(%)',
    title='그룹별 주요 장르 비율 (그룹 내 %)',
    labels={'genre': '장르', '비율(%)': '그룹 내 비율 (%)', 'response_group': '그룹'},
    category_orders={'genre': genre_order, 'response_group': GROUP_ORDER},
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=500, yaxis_range=[0, 55], legend_title_text='그룹')
fig.show()

In [7]:
# 장르 Lift: 초기 반응 그룹 비율 / 침묵 그룹 비율
genre_pivot = genre_dist.pivot(index='genre', columns='response_group', values='비율(%)')
genre_pivot.columns.name = None
genre_pivot['Lift (반응/침묵)'] = (genre_pivot['초기 반응 (≥10개)'] / genre_pivot['침묵 (<10개)']).round(2)
genre_pivot = genre_pivot.sort_values('Lift (반응/침묵)', ascending=False)

print('장르별 비율 및 Lift (초기 반응 / 침묵)')
display(genre_pivot.round(1))

장르별 비율 및 Lift (초기 반응 / 침묵)


,초기 반응 (≥10개),침묵 (<10개),Lift (반응/침묵)
genre,,,
Simulation,26.4,18.5,1.4
RPG,22.5,17.0,1.3
Adventure,51.5,42.9,1.2
Strategy,21.9,21.1,1.0
Sports,3.7,3.7,1.0
Action,44.1,45.7,1.0
Casual,44.3,53.8,0.8
Racing,3.2,4.1,0.8


**해석:**

- Lift > 1: 초기 반응 그룹에서 과대표현 → 해당 장르가 유저 반응을 이끌어내는 데 유리
- Lift < 1: 침묵 그룹에서 과대표현 → 해당 장르는 출시 후 반응이 없는 게임에 더 많이 분포

인디 개발사 관점에서, Lift가 높은 장르는 동일한 퀄리티라도 Steam 유저로부터 리뷰를 이끌어내기 더 쉬운 시장 구조를 갖고 있다.

## 분석 2. 태그 특이도 (Lift) 비교

각 게임의 투표 수 기준 상위 5개 태그를 추출해 그룹별 등장 비율을 계산하고, 전체 평균 대비 특이도(Lift)를 구한다.

$$Lift = \frac{\text{그룹 내 해당 태그 등장 비율}}{\text{전체 해당 태그 등장 비율}}$$

Lift > 1이면 해당 그룹에서 해당 태그가 기대보다 더 많이 등장 — 그 그룹의 "시그니처 태그".

In [8]:
MIN_GAMES_PER_TAG = 30
TOP_N_LIFT = 15

# 태그가 있는 게임만 사용
df_tag_base = df[df['top_tags'].map(len) > 0].copy()
total_games = df_tag_base['appid'].nunique()

df_tag = df_tag_base.explode('top_tags').rename(columns={'top_tags': 'tag'})
df_tag['tag'] = df_tag['tag'].astype(str).str.strip()
df_tag = df_tag[df_tag['tag'] != ''].copy()

# 전체 태그 등장 비율 (기준선)
total_tag_rate = (
    df_tag.groupby('tag')['appid'].nunique() / total_games
)

# 최소 게임 수 필터
valid_tags = total_tag_rate[total_tag_rate * total_games >= MIN_GAMES_PER_TAG].index
df_tag = df_tag[df_tag['tag'].isin(valid_tags)].copy()

# 그룹별 태그 등장 비율
group_games = df_tag_base.groupby('response_group')['appid'].nunique()
group_tag_rate = (
    df_tag.groupby(['response_group', 'tag'])['appid']
    .nunique()
    .reset_index(name='count')
)
group_tag_rate['group_rate'] = group_tag_rate.apply(
    lambda r: r['count'] / group_games[r['response_group']], axis=1
)
group_tag_rate['total_rate'] = group_tag_rate['tag'].map(total_tag_rate)
group_tag_rate['lift'] = (group_tag_rate['group_rate'] / group_tag_rate['total_rate']).round(3)

print(f'분석 태그 수 (최소 {MIN_GAMES_PER_TAG}개 이상 게임): {len(valid_tags)}개')

분석 태그 수 (최소 30개 이상 게임): 236개


In [9]:
def get_top_lift(group_name, top_n=TOP_N_LIFT, ascending=False):
    return (
        group_tag_rate[group_tag_rate['response_group'] == group_name]
        .sort_values('lift', ascending=ascending)
        .head(top_n)
    )

silence_top  = get_top_lift('침묵 (<10개)')
response_top = get_top_lift('초기 반응 (≥10개)')

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f'침묵 그룹 시그니처 태그 Top {TOP_N_LIFT}',
        f'초기 반응 그룹 시그니처 태그 Top {TOP_N_LIFT}',
    ],
    horizontal_spacing=0.18,
)

fig.add_trace(
    go.Bar(
        x=silence_top['lift'],
        y=silence_top['tag'],
        orientation='h',
        marker_color=GROUP_COLORS['침묵 (<10개)'],
        text=silence_top['lift'].map(lambda v: f'{v:.2f}'),
        textposition='outside',
        showlegend=False,
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Bar(
        x=response_top['lift'],
        y=response_top['tag'],
        orientation='h',
        marker_color=GROUP_COLORS['초기 반응 (≥10개)'],
        text=response_top['lift'].map(lambda v: f'{v:.2f}'),
        textposition='outside',
        showlegend=False,
    ),
    row=1, col=2,
)

fig.update_yaxes(autorange='reversed', row=1, col=1)
fig.update_yaxes(autorange='reversed', row=1, col=2)
fig.add_vline(x=1.0, line_dash='dot', line_color='gray', opacity=0.6, row=1, col=1)
fig.add_vline(x=1.0, line_dash='dot', line_color='gray', opacity=0.6, row=1, col=2)
fig.update_layout(
    title='그룹별 태그 특이도 (Lift) — 점선(1.0) 초과 시 해당 그룹에서 과대표현',
    height=560,
    width=1100,
)
fig.show()

**해석:**

- **침묵 그룹 시그니처 태그**: Lift가 높은 태그는 반응 없이 묻히는 게임에서 자주 선택된 태그다. 해당 태그를 가진 게임이 많아 경쟁이 치열하거나, 장르 특성상 유저 주목도가 낮은 카테고리일 수 있다.
- **초기 반응 그룹 시그니처 태그**: 리뷰를 이끌어낸 게임에서 과대표현된 태그다. 이 태그는 Steam 플랫폼에서 발견 가능성이 높거나 유저 관심도가 높은 카테고리와 연결될 가능성이 크다.

인디 개발사 관점에서, 초기 반응 그룹 시그니처 태그를 게임의 핵심 속성과 일치하는 범위 내에서 선택하면 출시 초기 노출과 첫 리뷰 확보에 유리하다.

## 분석 3. 출시 가격 비교

두 그룹의 출시 가격 분포를 비교한다. 무료 게임(price = 0)은 별도 처리한다.

In [10]:
df['price'] = pd.to_numeric(df['price'], errors='coerce')

# 무료/유료 분리
free_counts = df[df['price'] == 0].groupby('response_group')['appid'].count().reindex(GROUP_ORDER, fill_value=0)
df_paid = df[df['price'] > 0].copy()

print('무료 게임 수 (price = 0):')
for g, cnt in free_counts.items():
    total_g = len(df[df['response_group'] == g])
    print(f'  {g}: {cnt:,}개 ({cnt / total_g * 100:.1f}%)')

print(f'\n유료 게임 분석 대상: {len(df_paid):,}개')

무료 게임 수 (price = 0):
  침묵 (<10개): 0개 (0.0%)
  초기 반응 (≥10개): 0개 (0.0%)

유료 게임 분석 대상: 15,406개


In [11]:
fig = px.box(
    df_paid,
    x='response_group',
    y='price',
    color='response_group',
    color_discrete_map=GROUP_COLORS,
    category_orders={'response_group': GROUP_ORDER},
    title='그룹별 출시 가격 분포 (유료 게임, USD)',
    labels={'response_group': '그룹', 'price': '출시 가격 (USD)'},
    points='outliers',
)
fig.update_layout(showlegend=False, height=460, yaxis_range=[-1, 60])
fig.show()

In [12]:
price_bins   = [0, 5, 10, 15, 20, 30, 9999]
price_labels = ['$5 이하', '$5~10', '$10~15', '$15~20', '$20~30', '$30 초과']

df_paid = df_paid.copy()
df_paid['price_range'] = pd.cut(df_paid['price'], bins=price_bins, labels=price_labels, right=True)

price_dist = (
    df_paid.groupby(['response_group', 'price_range'], observed=True)['appid']
    .count()
    .reset_index(name='game_count')
)
paid_total = df_paid.groupby('response_group')['appid'].count()
price_dist['비율(%)'] = price_dist.apply(
    lambda r: r['game_count'] / paid_total[r['response_group']] * 100, axis=1
).round(1)

fig = px.bar(
    price_dist,
    x='price_range',
    y='비율(%)',
    color='response_group',
    color_discrete_map=GROUP_COLORS,
    barmode='group',
    text='비율(%)',
    title='그룹별 가격대 비율 (유료 게임, 그룹 내 %)',
    labels={'price_range': '가격대', '비율(%)': '그룹 내 비율 (%)', 'response_group': '그룹'},
    category_orders={'price_range': price_labels, 'response_group': GROUP_ORDER},
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=500, yaxis_range=[0, 55], legend_title_text='그룹')
fig.show()

In [13]:
price_stats = (
    df_paid.groupby('response_group')['price']
    .agg(중앙값='median', 평균='mean', Q1=lambda x: x.quantile(0.25), Q3=lambda x: x.quantile(0.75))
    .round(2)
    .reindex(GROUP_ORDER)
)
display(price_stats)

,중앙값,평균,Q1,Q3
response_group,,,,
침묵 (<10개),3.99,5.63,1.99,5.99
초기 반응 (≥10개),6.66,9.07,3.99,11.99


**해석:**

- 가격대 분포를 비교하면 어느 가격 구간에서 침묵 비율이 높고, 초기 반응 비율이 높은지 확인할 수 있다.
- 단, 가격 자체가 반응을 결정한다기보다 가격과 장르·퀄리티의 조합이 중요하다. 가격 데이터는 "기대치 설정"의 기준으로 해석해야 한다.
- 인디 개발사 관점에서, 시장 기대치보다 현저히 높은 가격 책정은 구매 전환율을 낮춰 리뷰 수 자체를 줄일 수 있다.

## 종합 해석 — 유저 첫 반응을 이끌어내기 위한 최소 기획 조건

세 가지 분석(장르·태그·가격)을 종합하면 다음과 같은 기획 조건을 도출할 수 있다.

In [14]:
# 장르별 Lift 요약
print('=== 장르별 Lift (초기 반응 / 침묵) 요약 ===')
display(genre_pivot[['초기 반응 (≥10개)', '침묵 (<10개)', 'Lift (반응/침묵)']].sort_values('Lift (반응/침묵)', ascending=False))

# 태그 Lift 요약
print('\n=== 초기 반응 그룹 시그니처 태그 Top 10 ===')
display(
    response_top[['tag', 'lift', 'count']]
    .rename(columns={'tag': '태그', 'lift': 'Lift', 'count': '게임 수'})
    .head(10)
    .reset_index(drop=True)
)

# 가격 요약
print('\n=== 그룹별 가격 통계 (USD) ===')
display(price_stats)

=== 장르별 Lift (초기 반응 / 침묵) 요약 ===


,초기 반응 (≥10개),침묵 (<10개),Lift (반응/침묵)
genre,,,
Simulation,26.4,18.5,1.43
RPG,22.5,17.0,1.32
Adventure,51.5,42.9,1.20
Strategy,21.9,21.1,1.04
Sports,3.7,3.7,1.00
Action,44.1,45.7,0.96
Casual,44.3,53.8,0.82
Racing,3.2,4.1,0.78



=== 초기 반응 그룹 시그니처 태그 Top 10 ===


,태그,Lift,게임 수
0,Roguelike Deckbuilder,1.808,96
1,Narrative,1.769,26
2,Turn-Based,1.737,42
3,Co-op,1.696,119
4,Gore,1.680,43
5,Online Co-Op,1.678,136
6,Open World Survival Craft,1.668,34
7,Martial Arts,1.627,27
8,Management,1.585,139
9,Fast-Paced,1.563,40



=== 그룹별 가격 통계 (USD) ===


,중앙값,평균,Q1,Q3
response_group,,,,
침묵 (<10개),3.99,5.63,1.99,5.99
초기 반응 (≥10개),6.66,9.07,3.99,11.99


### 최소 기획 조건 제안

| 조건 | 초기 반응 그룹의 특징 | 인디 개발사 실용 제안 |
|------|---------------------|---------------------|
| **장르** | Lift > 1인 장르에서 초기 반응 비율이 높음 | 시장에서 리뷰를 이끌어내기 쉬운 장르를 선택하거나, 해당 장르의 특성에 맞게 게임 포지셔닝 |
| **태그** | 특정 태그가 초기 반응 그룹에서 과대표현 | Steam 태그 설정 시 초기 반응 그룹 시그니처 태그를 게임 핵심 속성과 일치하는 범위에서 우선 선택 |
| **가격** | 초기 반응 그룹의 중앙값 가격이 기준점 | 장르 평균 대비 크게 벗어나지 않는 가격대 설정으로 구매 장벽 최소화 |

> **주의:** 위 조건은 필요 조건이지 충분 조건이 아니다. 장르·태그·가격을 최적화해도 게임 퀄리티와 출시 전 커뮤니티 구성이 뒷받침되지 않으면 첫 리뷰 확보는 어렵다. 이 분석은 "최소 진입 조건"을 찾는 데 목적이 있다.